# Fotos antiguas: reparar con una máscara y ver qué inventa el modelo

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial — IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento Digital de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Reparar dos fotos dañadas, una por vía clásica y otra con difusión, y distinguir qué se recupera de qué se inventa.

## Al terminar este cuaderno vas a poder:

1. Construir una máscara que marque el daño de una foto.
2. Reparar con `cv2.inpaint` y con un modelo de difusión, y comparar.
3. Explicar por qué una cara reconstruida es creíble pero no verificable.

## Preparación

Necesita **GPU** (T4). Usamos dos fotos dañadas de verdad (dominio público): un **retrato de 1860** con los ojos oscurecidos y un **negativo de vidrio de 1888** con emulsión desprendida. Ninguna tiene "original limpio": **no hay forma de comprobar** si la reparación es correcta.

In [ ]:
%pip install -q diffusers transformers accelerate

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/015_modelos_difusion'  # carpeta de la unidad en tu Drive
DATOS = f'{CARPETA}/datos'  # imágenes de ejemplo

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Sin GPU. Menú: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU')

print('GPU:', torch.cuda.get_device_name(0))
print('Memoria de la GPU:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

In [ ]:
import os

RESULTADOS = f'{CARPETA}/resultados'  # acá guardamos lo que generamos
os.makedirs(RESULTADOS, exist_ok=True)

## 1. El retrato: la máscara decide qué se inventa

El **inpainting** repara solo la zona que le marcás con una **máscara** (blanco = reparar, negro = conservar). Marcamos a mano un rectángulo sobre los ojos.

In [ ]:
import gc
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

retrato = Image.open(f'{DATOS}/danado_retrato_1860.jpg').convert('RGB')
retrato = retrato.crop((450, 330, 950, 830)).resize((512, 512))  # recorte centrado en la cara

mascara_ojos = np.zeros((512, 512), dtype=np.uint8)  # todo negro: no se toca
mascara_ojos[87:169, 164:358] = 255  # rectángulo blanco sobre los ojos
mascara_ojos = Image.fromarray(mascara_ojos)

retrato.save(f'{RESULTADOS}/retrato_original.png')

figura, ejes = plt.subplots(1, 2, figsize=(10, 5))
ejes[0].imshow(retrato)
ejes[0].set_title('retrato (1860)')
ejes[1].imshow(mascara_ojos, cmap='gray')
ejes[1].set_title('máscara: lo que se va a reparar')
ejes[0].axis('off')
ejes[1].axis('off')
plt.show()

Cargamos un modelo de **inpainting** de Stable Diffusion 1.5. Es el mismo modelo del cuaderno 03, entrenado además para rellenar zonas enmascaradas. Parámetros nuevos:

- `mask_image`: la máscara.
- `prompt`: qué debería haber en la zona. Es una **orden**, no una observación.

Generamos con tres *seeds* distintas.

In [ ]:
from diffusers import AutoPipelineForInpainting

pipe_reparar = AutoPipelineForInpainting.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-inpainting',
    torch_dtype=torch.float16,
    safety_checker=None,
).to('cuda')

semillas = [0, 1, 2]
variantes = []
for semilla in semillas:
    generador = torch.Generator('cuda').manual_seed(semilla)
    resultado = pipe_reparar(
        prompt='a nineteenth century black and white photograph of a man, clear eyes',
        image=retrato,
        mask_image=mascara_ojos,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generador,
    ).images[0]
    variantes.append(resultado)
    resultado.save(f'{RESULTADOS}/retrato_ojos_semilla{semilla}.png')

figura, ejes = plt.subplots(1, 4, figsize=(18, 5))
ejes[0].imshow(retrato.crop((120, 40, 400, 220)))
ejes[0].set_title('original')
for numero in range(len(variantes)):
    ejes[numero + 1].imshow(variantes[numero].crop((120, 40, 400, 220)))
    ejes[numero + 1].set_title(f'reparado, seed {semillas[numero]}')
for eje in ejes:
    eje.axis('off')
plt.show()

> **Para mirar.** Tres semillas dan tres miradas distintas, todas creíbles. **Ninguna es la persona real**: los ojos son una invención del modelo. Una foto restaurada así puede engañar a quien la ve.

## 2. El negativo de vidrio: una máscara automática

Acá el daño es grande y tiene un patrón: las zonas desprendidas quedan **negras**. Armamos la máscara con un umbral:

1. Marcar los píxeles muy oscuros (`gris < 22`).
2. Descartar manchas diminutas (que suelen ser sombras reales).
3. Ensanchar un poco la zona (`dilate`) para cubrir los bordes.

In [ ]:
puente = cv2.imread(f'{DATOS}/danado_puente_1888.jpg')  # OpenCV lee en BGR
gris = cv2.cvtColor(puente, cv2.COLOR_BGR2GRAY)

oscuros = (gris < 22).astype(np.uint8) * 255  # 255 donde el píxel es casi negro
cantidad, etiquetas, estadisticas, _ = cv2.connectedComponentsWithStats(oscuros)  # agrupa manchas conectadas
for numero in range(1, cantidad):  # la etiqueta 0 es el fondo
    area = estadisticas[numero][cv2.CC_STAT_AREA]
    if area < 25:
        oscuros[etiquetas == numero] = 0  # descartamos manchas chicas

mascara_puente = cv2.dilate(oscuros, np.ones((5, 5), np.uint8))
print(f'Zona marcada como dañada: {mascara_puente.mean() / 255:.1%}')

superpuesta = puente.copy()
superpuesta[mascara_puente > 0] = (0, 0, 255)  # rojo (en BGR)
figura, ejes = plt.subplots(1, 2, figsize=(14, 5))
ejes[0].imshow(cv2.cvtColor(puente, cv2.COLOR_BGR2RGB))
ejes[0].set_title('negativo dañado (1888)')
ejes[1].imshow(cv2.cvtColor(superpuesta, cv2.COLOR_BGR2RGB))
ejes[1].set_title('zona que se va a reparar (rojo)')
ejes[0].axis('off')
ejes[1].axis('off')
plt.show()

> **Para mirar.** La máscara agarra las manchas grandes pero **no** las grietas finas. Lo que reparás depende de lo que marcás: la máscara es una decisión tuya.

## 3. Reparar: clásico frente a difusión

`cv2.inpaint` rellena cada hueco propagando los píxeles de alrededor. `radius` es qué tan lejos mira. No inventa objetos, pero en huecos grandes deja manchas borrosas.

In [ ]:
puente_rgb = cv2.cvtColor(puente, cv2.COLOR_BGR2RGB)
reparado_clasico = cv2.inpaint(puente_rgb, mascara_puente, 5, cv2.INPAINT_TELEA)  # radio de 5 píxeles
Image.fromarray(reparado_clasico).save(f'{RESULTADOS}/puente_clasico.png')
Image.fromarray(reparado_clasico)

In [ ]:
foto_puente = Image.fromarray(puente_rgb).resize((768, 560))  # medidas múltiplos de 8
mascara_pil = Image.fromarray(mascara_puente).resize((768, 560), Image.NEAREST)  # NEAREST: sin grises intermedios

generador = torch.Generator('cuda').manual_seed(0)
reparado_difusion = pipe_reparar(
    prompt='an old black and white photograph of a stone bridge over a river, mountains and cloudy sky',
    image=foto_puente,
    mask_image=mascara_pil,
    height=560,
    width=768,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generador,
).images[0]
reparado_difusion.save(f'{RESULTADOS}/puente_difusion.png')

figura, ejes = plt.subplots(1, 3, figsize=(20, 5))
ejes[0].imshow(foto_puente)
ejes[0].set_title('dañado')
ejes[1].imshow(Image.fromarray(reparado_clasico).resize((768, 560)))
ejes[1].set_title('cv2.inpaint')
ejes[2].imshow(reparado_difusion)
ejes[2].set_title('difusión')
for eje in ejes:
    eje.axis('off')
plt.show()

del pipe_reparar
gc.collect()
torch.cuda.empty_cache()

> **Para mirar.** Fijate en el cielo: ¿qué rellenó cada método? El clásico estira lo que tiene alrededor; la difusión dibuja **nubes que nunca existieron**. En el cielo no importa, pero en una cara o en un documento sí.

## 4. Y el color

Una foto en blanco y negro no guarda el color: **hay que inventarlo**. Para poder comprobarlo usamos una foto que sí tiene el color real (Prokudin-Gorsky, 1909) y le quitamos el color.

In [ ]:
color_real = Image.open(f'{DATOS}/original_kirillov_1909.jpg').convert('RGB').resize((480, 415))
en_gris = color_real.convert('L').convert('RGB')  # blanco y negro, con tres canales
en_gris.save(f'{RESULTADOS}/kirillov_gris.png')

figura, ejes = plt.subplots(1, 2, figsize=(11, 5))
ejes[0].imshow(en_gris)
ejes[0].set_title('lo que tendría una foto en blanco y negro')
ejes[1].imshow(color_real)
ejes[1].set_title('el color real (lo sabemos por esta foto)')
ejes[0].axis('off')
ejes[1].axis('off')
plt.show()

Cualquier modelo que coloree la imagen de la izquierda tiene que **adivinar** rojos, violetas y verdes. Con una foto de archivo real, nadie puede decir cuál fue el color. Para el TPI, la sub-tarea C parte de esta idea.

> **Para probar.** Cambiá el `prompt` del puente por otro (por ejemplo, un cielo despejado) y mirá cómo cambia lo que dibuja la difusión en la zona enmascarada.

## Glosario

| Término | Qué significa acá |
|---|---|
| **Inpainting** | Rellenar una zona de una imagen según su contexto. |
| **Máscara** | Imagen en blanco y negro: blanco = reparar, negro = conservar. |
| **Umbral** | Valor de corte para decidir qué píxeles se marcan. |
| **Verosímil** | Que parece real. No es lo mismo que **verificable**. |
| **Emulsión** | Capa sensible a la luz de una fotografía antigua. |